# Milestone 4 Lab, notebook 3 of 7: Lineage

## Where we are

| Notebook | What you do |
|---|---|
| 1. A figure, traced by hand | Find every record behind one figure yourself, and add them up |
| 2. Run records | Give every pipeline run an id and a record, and keep every run |
| **3. Lineage** (you are here) | Save which records made each figure, and prove it adds up |
| 4. Two runs compared | Find what changed between two runs: settings, inputs, figures, records |
| 5. A trace of the analyst | Give the analyst the new tools, and record every step it takes |
| 6. Scoring an agent | Measure an agent with a confusion matrix, kappa, and the cost of mistakes |
| 7. Two experiments | Run two versions of the analyst on the same questions, and compare the scores |

Model requests in this notebook: **0**. A full run of this lab costs about 5 US cents.

## What this notebook is for

In notebook 1 you traced one figure by hand. It took a whole notebook, and it left nothing behind for the next person.

Here the pipeline does it for **every** figure, and saves the result as a table. That table is called **lineage**.

Then comes the important part: you prove the lineage is right. Lineage that cannot be checked is only a story.

No AI is used in this notebook.

> Run notebook 2 first. This notebook reads the runs it made.

In [ ]:
import pandas as pd
from lab_helpers import (
    ORDER_LINE_LINEAGE_SQL,
    add_row_numbers,
    load_result,
    load_source_tables,
    read_run_table,
    run_sql,
    show_code,
    show_doc,
)

## Step 1: The design

Lineage has two levels. Here they are, from the **HLD**.

In [ ]:
show_doc("HLD.md", "5. Lineage in one picture")

## Step 2: The first level, the steps

The steps say which table was made from which. The pipeline already knows this: milestone 2 gave every transformation a list of its inputs. `RUN-0002` saved that list.

In [ ]:
steps = read_run_table("RUN-0002", "lineage_steps")
steps

Read it as a path. `monthly_product_profit` is made from `sales` and four other tables. `sales` is made from `orders` and `order_items`. Each step also carries the fingerprint of the code that ran it.

The steps tell you **which tables** a figure passed through. They do not tell you **which records**. That is the second level.

## Step 3: The records the run kept

Lineage must only list records that counted. `RUN-0002` set 22 records aside, and saved which ones in its `finding_records` table.

In [ ]:
finding_records = read_run_table("RUN-0002", "finding_records")
set_aside = finding_records[finding_records["quarantined"]]

set_aside.groupby("source_table").size()

We load the source tables, number their rows as milestone 3 did, and drop the rows that were set aside.

In [ ]:
tables = load_source_tables()

kept = {}
for name, table in tables.items():
    numbered = add_row_numbers(table)
    rows_to_drop = set_aside[set_aside["source_table"] == name]["row_number"]
    kept[name] = numbered[~numbered["row_number"].isin(rows_to_drop)]
    print(name, len(table), "rows,", len(kept[name]), "kept")

## Step 4: Lineage for one term, by hand

Start small: only **gross sales**, which comes from order lines.

The query below returns one row for each order line that counts. It does not add anything up. It keeps every line, and says which product and month it belongs to and how much it adds.

In [ ]:
show_code(ORDER_LINE_LINEAGE_SQL, "sql")

Compare it with milestone 2's `sales` transformation. That query had the same join and the same rule about cancelled orders, followed by `SUM` and `GROUP BY`. **Lineage is the transformation without the adding up.**

In [ ]:
by_hand = run_sql(ORDER_LINE_LINEAGE_SQL, kept)

print(len(by_hand), "rows: one for each order line that counts")
by_hand.head()

Now filter it to the trail water bottle in October. In notebook 1 you found 250 order lines with gross sales of $20,280.00.

In [ ]:
lineage_key = load_result("answer_key_lineage", required_from="notebook 1")

bottle = by_hand[(by_hand["product_id"] == "P001") & (by_hand["month"] == "2026-10")]

print("Order lines:", len(bottle), "   by hand in notebook 1:", lineage_key["record_counts"]["order_items"])
print("Gross sales:", bottle["amount"].sum(), "   by hand in notebook 1:", lineage_key["gross_sales"])

## Step 5: The full lineage

Production has the same query for all nine terms. Here it is. It is longer, and every part has the same shape as the part you just ran.

In [ ]:
from retail_profit_stack.lineage.lineage import LINEAGE_SQL_FILE

show_code(LINEAGE_SQL_FILE.read_text(encoding="utf-8"), "sql")

`RUN-0002` ran that query and saved the result in its folder.

In [ ]:
lineage = read_run_table("RUN-0002", "figure_lineage")

print(len(lineage), "rows")
lineage.head()

Each row says: this record adds this amount to this term of this product's figures in this month.

Now ask the question from notebook 1 again. Which records are behind the trail water bottle's October figures?

In [ ]:
bottle_lineage = lineage[(lineage["product_id"] == "P001") & (lineage["month"] == "2026-10")]

records = bottle_lineage[["source_table", "row_number"]].drop_duplicates()
counts = records.groupby("source_table").size().to_dict()

print("From lineage:", counts)
print("By hand:     ", lineage_key["record_counts"])
print("Total:", len(records), "records")

The same 754 records you found by hand, in three lines.

## Step 6: The proof

How do we know the lineage query is right? It repeats the rules of the transformations, and a rule could have been copied wrongly.

The test is simple. **Add up each figure's lineage. The total must equal the figure.** Here it is for the trail water bottle in October.

In [ ]:
lineage_totals = bottle_lineage.groupby("term")["amount"].sum().round(2)

profit = read_run_table("RUN-0002", "monthly_product_profit")
bottle_profit = profit[(profit["product_id"] == "P001") & (profit["month"] == "2026-10")].iloc[0]

rows = []
for term, total in lineage_totals.items():
    rows.append({
        "term": term,
        "lineage adds up to": total,
        "profit table says": round(float(bottle_profit[term]), 2),
        "same?": "yes" if abs(total - bottle_profit[term]) < 0.005 else "NO",
    })

pd.DataFrame(rows)

All nine terms match. Production does this for every product and month, on every run. `lineage_differences` returns the totals that differ, so an empty table is the proof.

In [ ]:
from retail_profit_stack.lineage.lineage import lineage_differences

differences = lineage_differences(lineage, profit)

print("Totals that differ:", len(differences))

## Step 7: Break it on purpose

A check you have never seen fail is a check you cannot trust. Remove one row from the lineage and run the proof again.

In [ ]:
broken = lineage.drop(index=lineage.index[0])

lineage_differences(broken, profit)

The proof found the one figure whose lineage no longer adds up. The pipeline writes the result of this proof into the run record, as `lineage_adds_up`.

## Step 8: From terms to a metric

Lineage is saved per term. A metric is made of terms, each added or taken away.

In [ ]:
from retail_profit_stack.lineage.lineage import METRIC_TERMS

METRIC_TERMS

So the lineage totals alone are enough to rebuild the figure, with no profit table at all.

In [ ]:
t = lineage_totals

net_sales = t["gross_sales"] - t["discounts"] - t["refunds"]
gross_profit = net_sales - t["product_cost"]
contribution_profit = (
    gross_profit - t["fulfillment_cost"] - t["shipping_cost"] - t["payment_fees"] - t["return_cost"]
)

print("Rebuilt from lineage:", round(contribution_profit, 2))
print("By hand, notebook 1: ", lineage_key["contribution_profit"])

## Step 9: Any figure, in one line

Notebook 1 took a whole notebook for one figure. With lineage saved, any figure is one function call.

In [ ]:
from retail_profit_stack.lineage.lineage import record_counts

print("Sun hat, September:   ", record_counts(lineage, "2026-09", "P011"))
print("Camp stove, October:  ", record_counts(lineage, "2026-10", "P007"))

Look at the camp stove: no `product_costs` record. That is the missing cost from milestone 1, seen from a new side. Its lineage has no cost record, so its profit cannot be calculated.

## What we now know

- **Lineage is the transformation without the adding up.** It keeps one row per record, where the transformation keeps one row per product and month.
- It has two levels: the **steps** (which table from which) and the **records** (which records, and how much each adds).
- Lineage is **proved** on every run: its totals must equal the profit table. We broke it on purpose and the proof caught it.
- A quarantined record is not in lineage, because it did not count.
- Tracing a figure is now one line, for any product and month.

**Next:** notebook 4. With two runs saved, each with its lineage, we can ask what changed between them.